In [1]:
from collections import deque

def bfs(graph, start):
    visited = set()
    queue = deque([start])
    visited.add(start)


    while queue:
        node = queue.popleft()
        print(node, end=" ")

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

#     Adjacency list
graph = {
    "A": ["B", "C"],
    "B": ["C", "D"],
    "C": ["D", "E"],
    "D": ["E"],
    "E": ["F"],
    "F": [],
}
bfs(graph, 'A')
print("\t")
bfs(graph, 'B')
print("\t")
bfs(graph, 'C')
print("\t")
bfs(graph, 'E')

A B C D E F 	
B C D E F 	
C D E F 	
E F 

In [2]:
def bfs1(graph, start):
    visited = set()
    queue = deque([start])
    visited.add(start)

    while queue:
        node = queue.popleft()
        print(node, end=" ")

        for neg in graph.get(node, []):
            if neg not in visited:
                visited.add(neg)
                queue.append(neg)

graph1 = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1],
    5: [2]
}

bfs1(graph1, 0)
print("\t")
bfs1(graph1, 1)

0 1 2 3 4 5 	
1 0 3 4 2 5 

In [3]:
from collections import deque

def bfs(graph, start):
    """
    Breadth-First Search implementation using Queue
    graph: dict where key = node, value = list of neighbors
    start: starting node
    """
    # Track visited nodes
    visited = set()

    # Create queue and add starting node
    queue = deque([start])

    # Mark start as visited
    visited.add(start)

    # Store the order of visitation (optional but useful)
    traversal_order = []

    while queue:
        # Get the next node from front of queue
        current = queue.popleft()

        # Process current node
        traversal_order.append(current)
        print(current, end=" → ")

        # Visit all unvisited neighbors
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    print("END")
    return traversal_order

graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("BFS starting from node 'A':")
bfs(graph, 'A')
# Output: A → B → C → D → E → F → END

BFS starting from node 'A':
A → B → C → D → E → F → END


['A', 'B', 'C', 'D', 'E', 'F']

In [4]:
from collections import deque


class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def __repr__(self):
        return f"<Node state={self.state}, cost={self.path_cost}>"


def child_node(problem, parent_node, action):
    """Returns a child node resulting from applying action in parent_node's state"""
    next_state = problem.result(parent_node.state, action)
    path_cost = parent_node.path_cost + problem.step_cost(parent_node.state, action, next_state)
    return Node(next_state, parent_node, action, path_cost)


def solution(node):
    """Returns the sequence of actions from root to this node"""
    path = []
    current = node
    while current.parent is not None:
        path.append(current.action)
        current = current.parent
    path.reverse()
    return path


def breadth_first_search(problem):
    """
    Breadth-First Search - exactly following the classic pseudocode

    Returns:
        - list of actions (solution path) if goal found
        - None if no solution
    """
    node = Node(state=problem.initial_state, path_cost=0)

    if problem.goal_test(node.state):
        return solution(node)

    # FIFO queue
    frontier = deque([node])

    # Use set of states (assuming states are hashable)
    explored = set()

    while frontier:
        node = frontier.popleft()           # shallowest node first

        explored.add(node.state)

        for action in problem.actions(node.state):
            child = child_node(problem, node, action)

            # Important: check both explored and frontier
            if (child.state not in explored and
                child.state not in [n.state for n in frontier]):

                if problem.goal_test(child.state):
                    return solution(child)

                frontier.append(child)

    return None  # failure


# ────────────────────────────────────────────────
#   Minimal Problem interface you need to implement
# ────────────────────────────────────────────────

class Problem:
    def __init__(self, initial_state):
        self.initial_state = initial_state

    def actions(self, state):
        """Returns list of valid actions from this state"""
        raise NotImplementedError

    def result(self, state, action):
        """Returns the state that results from doing action in state"""
        raise NotImplementedError

    def goal_test(self, state):
        """Returns True if state is a goal state"""
        raise NotImplementedError

    def step_cost(self, state, action, next_state):
        """Cost of taking action in state to reach next_state (default = 1)"""
        return 1

In [1]:
#uniform cost search

from collections import defaultdict
import heapq
from typing import Any, Optional, List, Callable

# -------------------------------------------------------------------------
# Node class (same as in BFS/DFS versions)
# -------------------------------------------------------------------------
class Node:
    def __init__(self, state: Any, parent: Optional['Node'] = None,
                 action: Any = None, path_cost: float = 0.0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost          # g(n)

    def __lt__(self, other: 'Node') -> bool:
        # For heapq comparison (priority = path_cost)
        return self.path_cost < other.path_cost

    def __repr__(self) -> str:
        return f"<Node state={self.state}, g={self.path_cost:.1f}>"

    def solution(self) -> List[Any]:
        """Returns the sequence of actions from start to this node"""
        actions = []
        node = self
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        return actions[::-1]  # reverse to get start → goal order


# -------------------------------------------------------------------------
# Abstract Problem class (you implement this for your domain)
# -------------------------------------------------------------------------
class Problem:
    def __init__(self, initial_state: Any, goal_test: Callable[[Any], bool]):
        self.initial_state = initial_state
        self.goal_test = goal_test

    def actions(self, state: Any) -> List[Any]:
        raise NotImplementedError

    def result(self, state: Any, action: Any) -> Any:
        raise NotImplementedError

    def step_cost(self, state: Any, action: Any, next_state: Any) -> float:
        """Default uniform cost = 1, override for weighted graphs"""
        return 1.0


# -------------------------------------------------------------------------
# Uniform Cost Search (Dijkstra when step_cost ≥ 0)
# -------------------------------------------------------------------------
def uniform_cost_search(problem: Problem) -> Optional[Node]:
    """
    Uniform-Cost Search (priority queue by path cost g(n))
    Returns solution node or None if failure
    """
    # Create root node
    root = Node(state=problem.initial_state, path_cost=0.0)

    # Priority queue: (path_cost, node) — heapq uses tuple comparison
    frontier = []
    heapq.heappush(frontier, (root.path_cost, root))

    # Track best known path cost to each state
    best_cost: dict[Any, float] = defaultdict(lambda: float('inf'))
    best_cost[root.state] = 0.0

    # For cycle detection / avoiding re-expansion
    explored: set[Any] = set()

    while frontier:
        # Pop node with lowest path cost
        _, node = heapq.heappop(frontier)

        # Goal check
        if problem.goal_test(node.state):
            return node

        # Skip if we already expanded this state with better or equal cost
        if node.state in explored:
            continue

        explored.add(node.state)

        # Expand
        for action in problem.actions(node.state):
            child_state = problem.result(node.state, action)
            step = problem.step_cost(node.state, action, child_state)
            child_cost = node.path_cost + step

            # If we found a better path to this state
            if child_cost < best_cost[child_state]:
                child = Node(
                    state=child_state,
                    parent=node,
                    action=action,
                    path_cost=child_cost
                )

                # Update best known cost
                best_cost[child_state] = child_cost

                # Push to frontier (even if already present → lazy Dijkstra style)
                heapq.heappush(frontier, (child_cost, child))

    return None  # failure


# -------------------------------------------------------------------------
# Alternative version: more literal to pseudocode (checks frontier explicitly)
# -------------------------------------------------------------------------
def uniform_cost_search_literal(problem: Problem) -> Optional[Node]:
    """
    Closer to the exact pseudocode — checks if state is in frontier
    (more expensive due to linear search in frontier)
    """
    root = Node(state=problem.initial_state, path_cost=0.0)

    # Priority queue: list of nodes, we'll use heapq
    frontier: List[Node] = []
    heapq.heappush(frontier, root)

    explored: set[Any] = set()

    # Helper to find if a state exists in frontier and get its index
    def find_in_frontier(state: Any) -> Optional[int]:
        for i, node in enumerate(frontier):
            if node.state == state:
                return i
        return None

    while frontier:
        node = heapq.heappop(frontier)

        if problem.goal_test(node.state):
            return node

        explored.add(node.state)

        for action in problem.actions(node.state):
            child_state = problem.result(node.state, action)
            step = problem.step_cost(node.state, action, child_state)
            child_cost = node.path_cost + step

            child = Node(child_state, node, action, child_cost)

            idx = find_in_frontier(child_state)

            if child_state not in explored and idx is None:
                # Not in explored or frontier → add
                heapq.heappush(frontier, child)
            elif idx is not None:
                # In frontier → replace if better
                if child_cost < frontier[idx].path_cost:
                    frontier[idx] = child
                    # heapq doesn't support decrease-key easily → we just push
                    # and allow duplicates (lazy version is usually preferred)

    return None
# -------------------------------------------------------------------------
# Very simple example usage (pathfinding with different costs)
# -------------------------------------------------------------------------
if __name__ == "__main__":
    class SimpleGraphProblem(Problem):
        def __init__(self):
            super().__init__(initial_state='S', goal_test=lambda s: s == 'G')
            self.graph = {
                'S': [('A', 3), ('B', 5)],
                'A': [('C', 2), ('G', 10)],
                'B': [('G', 2)],
                'C': [('G', 1)],
            }

        def actions(self, state):
            return [a for a, _ in self.graph.get(state, [])]

        def result(self, state, action):
            for next_state, cost in self.graph.get(state, []):
                if next_state == action:
                    return next_state
            return state  # shouldn't happen

        def step_cost(self, state, action, next_state):
            for n, c in self.graph.get(state, []):
                if n == next_state:
                    return c
            return 1.0

    problem = SimpleGraphProblem()
    result = uniform_cost_search(problem)

    if result:
        print("Solution found!")
        print("Actions:", result.solution())
        print("Total cost:", result.path_cost)
        path = []
        node = result
        while node:
            path.append(node.state)
            node = node.parent
        print("Path:", " → ".join(reversed(path)))
    else:
        print("No solution")

Solution found!
Actions: ['A', 'C', 'G']
Total cost: 6.0
Path: S → A → C → G
